# ChakraModel Eval Harness — Kaggle Runner

Requires:
1. Dataset **ChakraModel_EvalHarness_Code** (the tiny code zip) attached
2. Dataset **ChakraModel_EvalHarness_Weights** (chakra_transformer_best.pth + combo1_best.pth) attached
3. Any polyp datasets you want scored (PolypGen, HyperKvasir, CVC-ClinicDB, etc.) attached — no code changes needed, `dataset_discovery.py` auto-scans `/kaggle/input`
4. Settings (right panel): **Accelerator = GPU T4 x2**, **Internet = On** (timm needs to fetch the ViT-Large pretrained backbone once before the checkpoint overwrites it), **Persistence = Files only** if you want outputs kept between sessions

In [1]:
!pip install -q timm albumentations opencv-python-headless psutil

In [2]:
import os, shutil, glob

WORK = '/kaggle/working'

# --- unpack the code bundle ---
code_zip = None
for p in glob.glob('/kaggle/input/**/ChakraModel_EvalHarness_Code.zip', recursive=True):
    code_zip = p
    break
assert code_zip, 'ChakraModel_EvalHarness_Code.zip not found under /kaggle/input — attach that dataset.'
print('Unpacking code from', code_zip)
shutil.unpack_archive(code_zip, WORK)

# --- stage weights where run_eval.py's defaults expect them ---
os.makedirs(f'{WORK}/weights/checkpoints', exist_ok=True)
wanted = {'chakra_transformer_best.pth', 'combo1_best.pth'}
found = set()
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f in wanted and f not in found:
            src = os.path.join(root, f)
            dst = f'{WORK}/weights/checkpoints/{f}'
            shutil.copy(src, dst)
            found.add(f)
            print('Linked', f, 'from', src)
missing = wanted - found
if missing:
    print('WARNING missing weights:', missing, '— attach the ChakraModel_EvalHarness_Weights dataset.')
else:
    print('Both checkpoints staged OK.')

AssertionError: ChakraModel_EvalHarness_Code.zip not found under /kaggle/input — attach that dataset.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(' -', torch.cuda.get_device_name(i))

In [ ]:
%cd /kaggle/working
!python eval_harness/run_eval.py

In [ ]:
# Sanity peek at the results file
import json, glob
res_files = sorted(glob.glob('/kaggle/working/eval_harness/results/evaluation_results_*.json'))
assert res_files, 'No results file produced.'
with open(res_files[-1]) as f:
    results = json.load(f)
for r in results:
    print(r.get('dataset'), '->', r.get('vit_large', {}).get('dice'), r.get('error', ''))